In [1]:
from oo_cqed_rhf import CQEDRHFCalculator
import numpy as np
import psi4
import sys
from typing import TextIO  # ← add this import
psi4.core.be_quiet()

In [2]:
# conversion factor
BOHR_TO_ANGSTROM = 0.52917721 


def geom_bohr_to_angstrom_string(geom_bohr: np.ndarray,
                                 symbols: list[str]) -> str:
    """
    Convert an (N,3) array in Bohr to a psi4 geometry input string in Angstrom,
    appending the fixed Psi4 directives at the end.

    Returns a string like:
        Li   x   y   z
        H   x   y   z
        0 1
        no_reorient
        nocom
        symmetry c1
    """
    # Conversion factor: 1 bohr = 0.529177210 Å
    bohr2ang = 0.52917721

    # Convert coordinates
    geom_ang = geom_bohr * bohr2ang

    # Format atomic lines
    lines = []
    for sym, (x, y, z) in zip(symbols, geom_ang):
        lines.append(f"{sym:2s}  {x:14.12f}  {y:14.12f}  {z:14.12f}")

    # Append fixed Psi4 directives
    suffix = """
0 1
no_reorient
nocom
symmetry c1
""".strip()

    # Combine and return full input block
    return "\n".join(lines + [suffix])







In [3]:

import numpy as np
import psi4

# ===========================
# Global Constants (Atomic Units conversion)
# ===========================
FS_TO_AU = 41.34137314       # fs → atomic units
AMU_TO_AU = 1822.8884850     # amu → atomic units
BOHR_TO_ANGSTROM = 0.52917721092

# ===========================
# Lambda vector along z-axis
# ===========================
lambda_vector = np.array([0.0, 0.05, 0.05])

# ===========================
# Psi4 Options
# ===========================
# psi4 options
psi4_options = {
    "basis": "6-31G",
    "save_jk": True,
    "scf_type": "pk",
    "e_convergence": 1e-12,
    "d_convergence": 1e-12,
}


psi4.set_options(psi4_options)

# ===========================
# Molecular Geometry (LiH)
# ===========================
mol_string = """
0  1
Li  0.000000000000  0.000000000000  0.000000000000
H   0.000000000000  0.000000000000  1.500000000000
no_reorient
nocom
symmetry c1
"""

mol = psi4.geometry(mol_string)

# Run SCF energy calculation
energy = psi4.energy("scf")
print(f"RHF energy at x(t0) (cavity-free): {energy: .12f}\n")

# ===========================
# Atomic Masses (hard-coded, Ruby’s values)
# ===========================
atom_masses = np.array([12789.3918753, 1837.17993072])
print("Atomic masses in atomic units:\n", atom_masses, "\n")

# ===========================
# Atomic Symbols from Geometry
# ===========================
symbols = [mol.symbol(i) for i in range(mol.natom())]
print("Atomic symbols:", symbols)


RHF energy at x(t0) (cavity-free): -7.976859130046

Atomic masses in atomic units:
 [12789.3918753   1837.17993072] 

Atomic symbols: ['LI', 'H']


In [4]:
# ===========================
# CQED-RHF Calculation
# ===========================

# Initialize calculator with desired geometry
calc = CQEDRHFCalculator(lambda_vector, mol_string, psi4_options)

# Compute energy, gradient, and coupling strength for given geometry
qed_rhf_energy, qed_rhf_grad, qed_rhf_g = calc.calc_force_and_energy(
    mol_string, 
    use_psi4_scf_grad=False
)

# Print results
print(f"\nInitial QED-RHF Energy: {qed_rhf_energy:.12f}\n")

# Pretty-print gradient
print("QED-RHF Gradient (a.u.):")
print(np.array2string(qed_rhf_grad, precision=12, suppress_small=False))
print()


Not Using Density Fitting!

Initial QED-RHF Energy: -7.968007513105

QED-RHF Gradient (a.u.):
[[ 1.014449897873e-17  4.840133539158e-04  2.002898795818e-02]
 [-1.014449897873e-17 -4.840133539158e-04 -2.002898795818e-02]]



In [5]:
# ===========================
# Initial Conditions
# ===========================

# Time step (atomic units)
dt = 15  

# Extract initial geometry (Bohr units)
x0_bohr = mol.geometry().to_array()
print("x(t0) in atomic units (Bohr):\n", x0_bohr, "\n")

# Initial positions & velocities
x_curr = np.copy(x0_bohr)
v_curr = np.array([
    [0.0,  4e-4 / 12789.3918753, 0.0],
    [0.0, -4e-4 / 1837.15264615, 0.0]
])

print("Initial velocities (a.u.):\n", v_curr, "\n")

# Expand atomic masses to column vector
print("Atomic masses (a.u., column vector):\n", atom_masses[:, None])


x(t0) in atomic units (Bohr):
 [[0.         0.         0.        ]
 [0.         0.         2.83458919]] 

Initial velocities (a.u.):
 [[ 0.00000000e+00  3.12759202e-08  0.00000000e+00]
 [ 0.00000000e+00 -2.17728233e-07  0.00000000e+00]] 

Atomic masses (a.u., column vector):
 [[12789.3918753 ]
 [ 1837.17993072]]


Step 1: Compute acceleration at initial position
$$ {\bf a}(t_0) = -\frac{{\bf g}(x(t_0))}{ M} $$

where ${\bf a}(t_0)$ denotes the timestep at the initial time $t0$ and ${\bf g}(x(t_0))$ denotes the gradient at the initiap position.

In [6]:
# ===========================
# Initial Acceleration
# ===========================

# Compute acceleration at t0 (a.u.)
a_t0 = -qed_rhf_grad / atom_masses[:, None]

print("Acceleration a(t0) [a.u.]:\n", a_t0, "\n")


Acceleration a(t0) [a.u.]:
 [[-7.93196352e-22 -3.78449076e-08 -1.56606257e-06]
 [ 5.52177759e-21  2.63454518e-07  1.09020285e-05]] 



Step 2: Compute update to initial position using

$$ {\bf x}(t_1) = {\bf x}(t_0) + {\bf v}(t_0) dt + \frac{1}{2} {\bf a}(t_0) dt^2 $$

In [7]:
# ===========================
# Position Update (t1)
# ===========================

# Compute position at t1 (Bohr units)
x_t1 = x_curr + v_curr * dt + 0.5 * a_t0 * dt**2

print("x(t1) in Bohr:\n", x_t1, "\n")
print("x(t1) in Angstrom:\n", x_t1 * BOHR_TO_ANGSTROM, "\n")


x(t1) in Bohr:
 [[-8.92345896e-20 -3.78841330e-06 -1.76182040e-04]
 [ 6.21199979e-19  2.63727098e-05  2.83581567e+00]] 

x(t1) in Angstrom:
 [[-4.72209112e-20 -2.00474199e-06 -9.32315203e-05]
 [ 3.28724873e-19  1.39558370e-05  1.50064903e+00]] 



Step 3: Compute updated acceleration 

$$ {\bf a}(t_1) = -\frac{{\bf g}(x(t_1))}{M} $$

In [8]:
# ===========================
# Recalculate at x(t1)
# ===========================

# Convert x(t1) to geometry string (Angstrom units)
geom_block_new = geom_bohr_to_angstrom_string(x_t1, symbols)
print("Updated geometry block:\n", geom_block_new, "\n")

# Compute energy, gradient, and coupling strength at new geometry
E_new, grad_new, g_new = calc.calc_force_and_energy(
    geom_block_new, 
    use_psi4_scf_grad=False
)

# Compute acceleration at t1
a_t1 = -grad_new / atom_masses[:, None]

# Print results
print(f"Energy at x(t1): {E_new:.12f}\n")
print("Gradient at x(t1) [a.u.]:\n", grad_new, "\n")
print("Acceleration a(t1) [a.u.]:\n", a_t1, "\n")


Updated geometry block:
 LI  -0.000000000000  -0.000002004742  -0.000093231520
H   0.000000000000  0.000013955837  1.500649022416
0 1
no_reorient
nocom
symmetry c1 

Energy at x(t1): -7.968035520369

Gradient at x(t1) [a.u.]:
 [[-1.07121368e-17  4.83306944e-04  1.98848664e-02]
 [ 1.07121368e-17 -4.83306944e-04 -1.98848664e-02]] 

Acceleration a(t1) [a.u.]:
 [[ 8.37579839e-22 -3.77896735e-08 -1.55479374e-06]
 [-5.83074995e-21  2.63070011e-07  1.08235813e-05]] 



Step 4: Update velocity using

$$ v(t_1) = v(t_0) + \frac{1}{2} (a(t_0) + a(t_1)) dt $$

In [9]:
# ===========================
# Velocity Update (t1)
# ===========================

# Compute velocity at t1 (a.u.)
v_t1 = v_curr + 0.5 * (a_t0 + a_t1) * dt

print("Velocity v(t1) [a.u.]:\n", v_t1, "\n")


Velocity v(t1) [a.u.]:
 [[ 3.32876150e-22 -5.35983438e-07 -2.34064223e-05]
 [-2.31729264e-21  3.73120573e-06  1.62942074e-04]] 



In [10]:
# ===========================
# Position Update (t2)
# ===========================

# Compute position at t2 (Bohr units)
x_t2 = x_t1 + v_t1 * dt + 0.5 * a_t1 * dt**2

print("x(t2) in Bohr:\n", x_t2, "\n")
print("x(t2) in Angstrom:\n", x_t2 * BOHR_TO_ANGSTROM, "\n")


x(t2) in Bohr:
 [[ 9.98628450e-21 -1.60795032e-05 -7.02192670e-04]
 [-6.95187791e-20  1.11936172e-04  2.83947745e+00]] 

x(t2) in Angstrom:
 [[ 5.28451418e-21 -8.50890663e-06 -3.71584359e-04]
 [-3.67877536e-20  5.92340713e-05  1.50258676e+00]] 



In [11]:
# Convert x(t1) to geometry string (Angstrom units)
geom_block_new = geom_bohr_to_angstrom_string(x_t2, symbols)
print("Updated geometry block:\n", geom_block_new, "\n")

# Compute energy, gradient, and coupling strength at new geometry
E_new, grad_new, g_new = calc.calc_force_and_energy(
    geom_block_new, 
    use_psi4_scf_grad=False
)

# Compute acceleration at t1
a_t2 = -grad_new / atom_masses[:, None]

# Print results
print(f"Energy at x(t1): {E_new:.12f}\n")
print("Gradient at x(t1) [a.u.]:\n", grad_new, "\n")
print("Acceleration a(t1) [a.u.]:\n", a_t2, "\n")

Updated geometry block:
 LI  0.000000000000  -0.000008508907  -0.000371584358
H   -0.000000000000  0.000059234071  1.502586755060
0 1
no_reorient
nocom
symmetry c1 

Energy at x(t1): -7.968117943472

Gradient at x(t1) [a.u.]:
 [[-1.76695437e-18  4.81232951e-04  1.94565866e-02]
 [ 1.76695437e-18 -4.81232951e-04 -1.94565866e-02]] 

Acceleration a(t1) [a.u.]:
 [[ 1.38157810e-22 -3.76275085e-08 -1.52130663e-06]
 [-9.61775351e-22  2.61941111e-07  1.05904633e-05]] 



In [12]:
# ===========================
# Velocity Update (t1)
# ===========================

# Compute velocity at t1 (a.u.)
v_t2 = v_t1 + 0.5 * (a_t1 + a_t2) * dt

print("Velocity v(t1) [a.u.]:\n", v_t2, "\n")


Velocity v(t1) [a.u.]:
 [[ 7.65090851e-21 -1.10161230e-06 -4.64771751e-05]
 [-5.32612324e-20  7.66878914e-06  3.23547408e-04]] 

